# Week 2: Advanced Pandas Data Wrangling
## Notebook 09: Custom Rolling Windows & Numba JIT Acceleration

### 1. Conceptual Intuition & Theoretical Background
Standard Pandas `.rolling().apply(custom_func)` is notoriously slow because it converts each sliding window slice into a temporary `pd.Series`, incurring Python function call overhead millions of times.

**Numba JIT (`engine='numba'`)** compiles numerical Python functions into LLVM machine code. Furthermore, rolling variance and standard deviation can be computed using **Welford's Algorithm** in a numerically stable single pass:

$$M_k = M_{k-1} + \frac{x_k - M_{k-1}}{k}$$
$$S_k = S_{k-1} + (x_k - M_{k-1})(x_k - M_k)$$
$$\sigma^2_k = \frac{S_k}{k - 1}$$

---

### 2. Scratch Python Implementation: Welford Rolling Variance

In [ ]:
import numpy as np
import time

# Generate synthetic financial tick series
n_ticks = 50000
np.random.seed(42)
returns = np.random.normal(0.0002, 0.015, n_ticks)
window = 100

# 1. Naive Rolling Volatility (Python slice simulation)
def naive_rolling_std(arr, w):
    n = len(arr)
    result = np.full(n, np.nan)
    for i in range(w - 1, n):
        slice_window = arr[i - w + 1 : i + 1]
        result[i] = np.std(slice_window, ddof=1)
    return result

# 2. Optimized One-Pass Sliding Window Algorithm
def fast_sliding_variance(arr, w):
    n = len(arr)
    result = np.full(n, np.nan)
    
    # Initialize first window sum and sum of squares
    window_sum = np.sum(arr[:w])
    window_sq_sum = np.sum(arr[:w] ** 2)
    
    variance = (window_sq_sum - (window_sum ** 2) / w) / (w - 1)
    result[w - 1] = np.sqrt(max(0.0, variance))
    
    # Slide window: subtract leaving element, add entering element
    for i in range(w, n):
        leaving = arr[i - w]
        entering = arr[i]
        
        window_sum += entering - leaving
        window_sq_sum += entering ** 2 - leaving ** 2
        
        var = (window_sq_sum - (window_sum ** 2) / w) / (w - 1)
        result[i] = np.sqrt(max(0.0, var))
        
    return result

# Benchmark
t0 = time.perf_counter()
std_fast = fast_sliding_variance(returns, window)
t_fast = time.perf_counter() - t0

t0 = time.perf_counter()
std_naive = naive_rolling_std(returns[:5000], window) # tiny subset
t_naive = time.perf_counter() - t0

print(f"Fast O(N) Sliding Window Volatility Time: {t_fast * 1000:.2f} ms for {n_ticks} points")
print(f"Naive O(N*W) Sliding Window Time:        {t_naive * 1000:.2f} ms for 5,000 points")
print(f"Speedup Factor: > {((t_naive * 10) / t_fast):.1f}x")

### 3. Step-by-Step Code Walkthrough
1. **Naive $O(N \times W)$**: Re-evaluates all $W$ elements from scratch at every step.
2. **Sliding $O(N)$ Formulation**: Updates running sum and sum-of-squares in $O(1)$ constant time per step by adding entering elements and subtracting exiting elements.
3. **Numerical Precision**: Protects against floating-point cancellation using bounded clamping.

---

### 4. Industry Applications & Trade-offs
- **High-Frequency Trading (HFT):** Calculating rolling Sharpe ratios, Bollinger Bands, and real-time VWAP on millions of trades per second.
- **Trade-off:** Naive sliding sum-of-squares can accumulate floating-point rounding errors over long sequences without periodic re-centering.

---

### 5. Top Interview Questions & Answers
**Q1: Why does `pandas.Series.rolling().apply()` become exponentially faster when passing `engine='numba'`?**
> **Answer:** Numba compiles the custom Python loop into machine code and avoids creating intermediate Series objects for each window, executing in native C-speed.

**Q2: How does Welford's algorithm avoid catastrophic numerical cancellation?**
> **Answer:** It updates deviations relative to the running mean incrementally rather than subtracting two large similar numbers ($\sum x^2 - (\sum x)^2 / n$), which causes floating point precision loss.